# Multi-Hop Question Answering with Retrieval Chaining

## 1. Project Overview

**Task:** Build a RAG system that answers **multi-hop questions** — questions that require combining evidence from **multiple documents** through a chain of reasoning steps. Each hop retrieves new evidence based on what was learned in the previous hop.

**Why this matters:** Most real questions aren't answerable from a single passage. "Which caching strategy is best for the deployment pattern used by Netflix?" requires: (1) finding what deployment pattern Netflix uses, (2) understanding that pattern's characteristics, and (3) matching a caching strategy to those characteristics. Standard single-retrieval RAG fails because no single passage contains all the information.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — LLM for reasoning, sub-question generation, and answer synthesis
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — embedding model
- `ChromaDB` — vector store

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **why single-hop retrieval fails** on complex questions |
| 2 | Build a **question decomposer** that breaks multi-hop questions into sub-questions |
| 3 | Implement **iterative retrieval** — each hop's answer guides the next hop's query |
| 4 | Design **reasoning chains** that accumulate evidence across hops |
| 5 | Build a **chain-of-thought synthesizer** that produces final answers with cited reasoning |
| 6 | Compare **single-hop vs multi-hop** retrieval on complex questions |

## 3. The Multi-Hop Problem

### Why Single-Hop Retrieval Fails

```
QUESTION: "What monitoring approach is recommended for the data store
           used by event-sourced architectures?"

SINGLE-HOP RETRIEVAL:
  Query: full question → Retrieves passages about monitoring OR event sourcing
  Problem: No single passage connects event sourcing → its data store → monitoring

MULTI-HOP RETRIEVAL:
  Hop 1: "What data store does event sourcing use?"
         → Event sourcing stores events in an append-only log (event store)
  Hop 2: "What monitoring is recommended for append-only event stores?"
         → Time-series monitoring, log-based metrics, stream lag monitoring
  Hop 3: Synthesize: "For event-sourced architectures, monitor the event
         store using time-series metrics on write throughput and consumer lag..."
```

### Types of Multi-Hop Questions

| Type | Example | Hops Needed |
|------|---------|-------------|
| **Bridge** | "What persistence is used by the cache in the auth flow?" | A→B→C (chain) |
| **Comparison** | "How does K8s scaling differ from the scaling in event-driven systems?" | A + B (parallel, then compare) |
| **Compositional** | "Which deployment strategy works best with the testing approach used in microservices?" | A→B, then match |
| **Inferential** | "Would Redis or Kafka be better for this architecture's caching layer?" | A + B + reasoning |

## 4. Pipeline Architecture

```
  Complex Question
       │
       ▼
  ┌─────────────────┐
  │ 1. DECOMPOSE    │  Break into sub-questions
  │    question      │  [Q1, Q2, Q3]
  └────────┬────────┘
           │
           ▼
  ┌─────────────────┐
  │ 2. HOP 1        │  Retrieve for Q1
  │    retrieve +   │  Answer Q1 from evidence
  │    answer Q1    │  → intermediate fact F1
  └────────┬────────┘
           │ F1 feeds into Q2
           ▼
  ┌─────────────────┐
  │ 3. HOP 2        │  Retrieve for Q2 (informed by F1)
  │    retrieve +   │  Answer Q2 from evidence + F1
  │    answer Q2    │  → intermediate fact F2
  └────────┬────────┘
           │ F1 + F2
           ▼
  ┌─────────────────┐
  │ 4. SYNTHESIZE   │  Combine F1, F2, ... into final answer
  │    final answer │  With full reasoning chain + citations
  └─────────────────┘
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers

print("Dependencies: langchain, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports

## 7. Configuration

In [ ]:
import os
import re
import json
import time
import textwrap
import warnings
from collections import Counter

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 3
MAX_HOPS = 4
TEMP_REASON = 0.1
TEMP_ANSWER = 0.2

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Max hops: {MAX_HOPS}")
print(f"Top-k per hop: {TOP_K}")

## 8. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Knowledge Base

## 9. Document Corpus

30 passages covering interconnected topics. The key is that **multi-hop questions require linking facts across multiple passages** that would never appear in the same single-hop retrieval.

In [ ]:
PASSAGES = [
    # Docker cluster
    {"id": "P01", "topic": "docker",
     "text": "Docker containers share the host OS kernel using Linux namespaces and cgroups for isolation. Images are built from Dockerfiles with a layered union file system. Each instruction creates a read-only layer. Containers start in under 2 seconds and use 50-100 MB of memory. Docker Hub is the default public registry for sharing images."},
    {"id": "P02", "topic": "docker",
     "text": "Docker Compose orchestrates multi-container applications using a YAML file. It defines services, networks, and volumes together. Environment variables, health checks, and dependency ordering via 'depends_on' are supported. Docker Compose is commonly used in development environments. In production, teams typically migrate to Kubernetes for orchestration."},

    # Kubernetes cluster
    {"id": "P03", "topic": "kubernetes",
     "text": "Kubernetes deploys containerized applications across clusters. The control plane includes the API server, etcd, scheduler, and controller manager. Pods are the smallest unit — one or more containers sharing network and storage. Deployments ensure a desired number of pod replicas run. If a pod crashes, the controller creates a replacement automatically."},
    {"id": "P04", "topic": "kubernetes",
     "text": "Kubernetes Horizontal Pod Autoscaler (HPA) adjusts replica count based on CPU utilization or custom metrics. Vertical Pod Autoscaler adjusts resource requests. Cluster Autoscaler adds or removes nodes. HPA checks metrics every 15 seconds by default. Scaling decisions use a stabilization window to prevent flapping."},
    {"id": "P05", "topic": "kubernetes",
     "text": "Kubernetes Services provide stable endpoints. ClusterIP is internal-only. NodePort exposes on each node. LoadBalancer provisions cloud load balancers. Ingress manages HTTP routing with path and host rules. Service mesh solutions like Istio add mutual TLS, traffic management, and observability."},

    # Database cluster
    {"id": "P06", "topic": "postgresql",
     "text": "PostgreSQL uses MVCC for concurrent access — readers never block writers. Write-Ahead Logging (WAL) ensures durability by recording changes before applying them. B-tree indexes handle equality and range queries. GiST indexes support full-text search and spatial queries. PostgreSQL can handle thousands of concurrent connections with connection pooling via PgBouncer."},
    {"id": "P07", "topic": "databases",
     "text": "Database sharding splits data across multiple servers. Hash-based sharding distributes evenly but makes range queries expensive. Range-based sharding supports range queries but can create hot spots. Cross-shard queries need scatter-gather operations. Vitess is a popular sharding middleware for MySQL, while Citus extends PostgreSQL with distributed tables."},
    {"id": "P08", "topic": "databases",
     "text": "Database replication copies data across multiple servers. Synchronous replication waits for all replicas to confirm writes — strong consistency but higher latency. Asynchronous replication is faster but risks data loss if the primary fails before replicas catch up. PostgreSQL supports both streaming replication and logical replication for different use cases."},

    # Caching cluster
    {"id": "P09", "topic": "redis",
     "text": "Redis is an in-memory store achieving sub-millisecond read/write latency. Data structures include strings, lists, sets, sorted sets, hashes, and streams. RDB persistence creates periodic snapshots. AOF logs every write for durability. Redis Sentinel provides automatic failover. Redis Cluster partitions data across nodes using 16384 hash slots."},
    {"id": "P10", "topic": "caching",
     "text": "Cache-aside (lazy loading) populates cache only on miss — first request is slow, subsequent ones are fast. Write-through updates cache and database together for consistency. Write-behind batches database writes asynchronously for performance. TTL-based expiration is simplest but can serve stale data. Each strategy has different consistency and performance tradeoffs."},
    {"id": "P11", "topic": "caching",
     "text": "Cache stampede occurs when a popular key expires and many requests simultaneously attempt to regenerate it, overwhelming the database. Solutions: mutex locking (one request regenerates, others wait), probabilistic early expiration (XFetch refreshes before TTL), and background refresh (separate process pre-updates hot keys). For high-traffic systems, cache warming during deployment prevents cold-start performance dips."},

    # API cluster
    {"id": "P12", "topic": "rest_api",
     "text": "REST APIs use HTTP methods semantically: GET reads, POST creates, PUT replaces, PATCH updates, DELETE removes. Resources are identified by URI nouns. Status codes communicate results: 2xx success, 4xx client error, 5xx server error. HATEOAS includes links in responses for discoverable navigation. REST is stateless — each request contains all information needed."},
    {"id": "P13", "topic": "graphql",
     "text": "GraphQL provides a single endpoint where clients specify exact data requirements through typed queries. Schemas define types and their relationships. Resolvers fetch data for each field. GraphQL eliminates over-fetching and under-fetching problems common in REST. The N+1 query problem is solved with DataLoader batching. Subscriptions enable real-time updates via WebSockets."},
    {"id": "P14", "topic": "api",
     "text": "API rate limiting protects services from abuse and overload. Token bucket allows bursts up to bucket size then throttles. Sliding window counts requests in a moving time frame. Headers communicate limits: X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Reset. Exceeding limits returns HTTP 429. Different API tiers can have different rate limits."},

    # Security cluster
    {"id": "P15", "topic": "oauth",
     "text": "OAuth 2.0 delegates authorization without exposing credentials. The authorization code flow redirects users to an auth server, collects consent, returns a code exchanged for tokens. Access tokens expire after 1 hour typically. Refresh tokens are long-lived for obtaining new access tokens. PKCE prevents code interception in public clients like mobile apps."},
    {"id": "P16", "topic": "jwt",
     "text": "JWT tokens contain three base64-encoded parts: header (algorithm), payload (claims like sub, exp, iat), and signature. JWTs are stateless — no server-side session storage needed. Common algorithms: HS256 (symmetric HMAC) and RS256 (asymmetric RSA). JWTs cannot be revoked before expiration without maintaining a blacklist, which partially defeats their stateless advantage."},
    {"id": "P17", "topic": "security",
     "text": "SQL injection inserts malicious SQL via unsanitized user input. Prevention: parameterized queries separate logic from data. XSS injects scripts into web pages — prevented by output encoding and Content Security Policy headers. CSRF tricks authenticated users into unwanted actions — prevented by anti-CSRF tokens. Defense in depth combines input validation, output encoding, and security headers."},

    # Messaging cluster
    {"id": "P18", "topic": "kafka",
     "text": "Apache Kafka is a distributed streaming platform. Producers publish to topics partitioned across brokers. Each partition is an ordered, append-only commit log. Consumer groups enable parallel processing — each partition assigned to one consumer. Messages are retained for a configurable period (default 7 days) regardless of consumption. Kafka achieves millions of messages per second throughput."},
    {"id": "P19", "topic": "rabbitmq",
     "text": "RabbitMQ implements AMQP for message queuing. Exchanges route messages to queues via routing keys and bindings. Direct exchanges match exact keys. Topic exchanges use wildcard patterns. Fanout exchanges broadcast to all bound queues. Messages are consumed and acknowledged — unacknowledged messages are redelivered. Dead letter queues capture failed messages after max retries."},
    {"id": "P20", "topic": "messaging",
     "text": "Message delivery guarantees: at-most-once (may lose messages), at-least-once (may duplicate), exactly-once (requires idempotent processing). Kafka supports exactly-once within its ecosystem via idempotent producers and transactions. Outbox pattern: write events to a database table atomically with business data, then publish asynchronously. This prevents dual-write inconsistency."},

    # Architecture cluster
    {"id": "P21", "topic": "microservices",
     "text": "Microservices decompose applications into independently deployable services. Each service owns its data store (database per service). Benefits: independent scaling, fault isolation, technology diversity. Challenges: distributed transactions, eventual consistency, operational overhead. Communication: synchronous (REST, gRPC) or asynchronous (events via Kafka or RabbitMQ)."},
    {"id": "P22", "topic": "event_sourcing",
     "text": "Event sourcing stores every state change as an immutable event rather than overwriting current state. The current state is rebuilt by replaying events. Benefits: complete audit trail, temporal queries, event-driven integrations. CQRS (Command Query Responsibility Segregation) separates write models (commands produce events) from read models (optimized projections). Event stores are typically append-only logs."},
    {"id": "P23", "topic": "architecture",
     "text": "The CAP theorem: distributed systems cannot simultaneously guarantee Consistency, Availability, and Partition tolerance. During partitions, choose CP (reject requests for consistency — HBase, MongoDB majority) or AP (accept writes risking conflicts — Cassandra, DynamoDB). PACELC extends this: without partitions, trade off Latency vs Consistency."},

    # Deployment cluster
    {"id": "P24", "topic": "deployment",
     "text": "Blue-green deployment maintains two identical environments. 'Blue' serves production traffic while 'green' receives the update. After testing green, traffic switches instantly. Rollback is fast — switch back to blue. Requires double infrastructure. Netflix popularized the canary deployment pattern as part of their continuous delivery pipeline."},
    {"id": "P25", "topic": "deployment",
     "text": "Canary deployment routes a small percentage (5-10%) of traffic to the new version. If error rates and latency remain healthy, traffic gradually increases to 100%. Automated canary analysis compares metrics between canary and baseline. Netflix's Spinnaker platform automates canary analysis with statistical comparison of metrics across both versions."},
    {"id": "P26", "topic": "ci_cd",
     "text": "CI/CD pipelines automate build, test, and deploy. Continuous Integration runs linting, unit tests, integration tests, and security scanning on every commit. Continuous Delivery keeps the main branch deployable. Pipeline stages: build → test → security scan → staging → production. Feature flags decouple deployment from release — code is deployed but toggled off until ready."},

    # Monitoring cluster
    {"id": "P27", "topic": "monitoring",
     "text": "The four golden signals (Google SRE): latency, traffic, errors, saturation. Prometheus scrapes metrics endpoints on a schedule (pull model). Metric types: counters (only increase), gauges (up/down), histograms (value distributions). PromQL queries time-series data. Grafana dashboards visualize Prometheus metrics. AlertManager routes and deduplicates alerts."},
    {"id": "P28", "topic": "tracing",
     "text": "Distributed tracing propagates trace IDs across service boundaries. Each unit of work is a span with start time, duration, and metadata. Parent-child span relationships form call trees. OpenTelemetry provides vendor-neutral instrumentation. Jaeger and Zipkin store and visualize traces. Traces reveal latency bottlenecks across microservice call chains."},

    # Testing
    {"id": "P29", "topic": "testing",
     "text": "The testing pyramid: many unit tests (fast, isolated with mocks), fewer integration tests (real component interactions), minimal end-to-end tests (full user flows). Contract testing verifies API compatibility between services. Chaos engineering intentionally injects failures (killed pods, network delays) to verify system resilience. Netflix's Chaos Monkey randomly terminates production instances."},

    # Load balancing
    {"id": "P30", "topic": "load_balancing",
     "text": "Layer 4 load balancers route by IP/TCP — fast but limited visibility. Layer 7 balancers inspect HTTP headers and URLs for content-based routing, SSL termination, and request modification. Algorithms: round-robin, least connections, weighted distribution, IP hash (session affinity). Health checks remove unhealthy backends. Cloud providers offer managed load balancers integrated with auto-scaling."},
]

print(f"Corpus: {len(PASSAGES)} passages, {len(set(p['topic'] for p in PASSAGES))} topics")
print(f"Avg length: {sum(len(p['text'].split()) for p in PASSAGES) // len(PASSAGES)} words")

## 10. Build the Retriever

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("kb")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="kb", metadata={"hnsw:space": "cosine"},
)

texts = [p["text"] for p in PASSAGES]
metas = [{"passage_id": p["id"], "topic": p["topic"]} for p in PASSAGES]
ids = [p["id"] for p in PASSAGES]
vecs = embeddings.embed_documents(texts)
collection.add(documents=texts, embeddings=vecs, metadatas=metas, ids=ids)

print(f"Indexed {collection.count()} passages")


def retrieve(query: str, top_k: int = TOP_K, exclude_ids: list = None) -> list[dict]:
    """Retrieve top-k passages, optionally excluding already-seen passage IDs."""
    fetch_k = top_k + (len(exclude_ids) if exclude_ids else 0) + 2
    qv = embeddings.embed_query(query)
    results = collection.query(query_embeddings=[qv], n_results=min(fetch_k, 30))
    hits = []
    for i in range(len(results["documents"][0])):
        pid = results["metadatas"][0][i]["passage_id"]
        if exclude_ids and pid in exclude_ids:
            continue
        hits.append({
            "passage_id": pid,
            "topic": results["metadatas"][0][i]["topic"],
            "text": results["documents"][0][i],
            "score": round(1 - results["distances"][0][i], 4),
        })
        if len(hits) >= top_k:
            break
    return hits


# Quick test
test = retrieve("Redis caching strategies")
print(f"\nTest: {[h['passage_id'] for h in test]}")

---

# Part B — Single-Hop Baseline

## 11. Standard Single-Hop RAG

First, let's see how standard single-hop RAG handles multi-hop questions.

In [ ]:
ANSWER_SYSTEM = """Answer the question using ONLY the provided passages.
Cite passages as [PXX]. If information is insufficient, say so."""


def single_hop_rag(question: str) -> dict:
    """Standard single-retrieval RAG."""
    t0 = time.time()
    passages = retrieve(question, top_k=5)
    context = "\n\n".join(f"[{p['passage_id']}] {p['text']}" for p in passages)
    prompt = f"QUESTION: {question}\n\nPASSAGES:\n{context}\n\nAnswer:"
    answer = ask(prompt, system=ANSWER_SYSTEM, temperature=TEMP_ANSWER)
    return {
        "method": "single_hop",
        "question": question,
        "answer": answer,
        "passages_used": [p["passage_id"] for p in passages],
        "time_s": time.time() - t0,
    }


# Test with a multi-hop question
HARD_Q = "What caching strategy works best with the deployment pattern Netflix uses?"

single = single_hop_rag(HARD_Q)
print(f"Q: {HARD_Q}")
print(f"Passages retrieved: {single['passages_used']}")
print(f"\nSingle-hop answer:")
wrap_print(single["answer"])

---

# Part C — Multi-Hop Pipeline

## 12. Question Decomposer

The first component breaks a complex question into ordered sub-questions. Each sub-question should be answerable independently, and later sub-questions can depend on answers from earlier ones.

In [ ]:
DECOMPOSE_SYSTEM = """You decompose complex questions into simpler sub-questions.
Each sub-question should be independently answerable from a knowledge base.
Order matters: later sub-questions can depend on earlier answers."""

DECOMPOSE_PROMPT = """Break this complex question into 2-4 simpler sub-questions.
Each sub-question should target ONE specific fact needed for the final answer.
Order them so earlier answers inform later questions.

QUESTION: {question}

Return ONLY a JSON list:
["sub-question 1", "sub-question 2", ...]"""


def decompose_question(question: str) -> list[str]:
    """Break a multi-hop question into sub-questions."""
    resp = ask(
        DECOMPOSE_PROMPT.format(question=question),
        system=DECOMPOSE_SYSTEM,
        temperature=TEMP_REASON,
    )
    result = parse_json(resp)
    if isinstance(result, list) and len(result) >= 2:
        return [str(q) for q in result]
    # Fallback: simple split
    return [question]


# Demo decompositions
test_questions = [
    "What caching strategy works best with the deployment pattern Netflix uses?",
    "How should you monitor a Kubernetes cluster running event-sourced microservices?",
    "What testing approach is recommended for APIs that use JWT authentication?",
]

print("QUESTION DECOMPOSITION")
print("=" * 90)
for q in test_questions:
    subs = decompose_question(q)
    print(f"\n  Q: {q}")
    for i, s in enumerate(subs, 1):
        print(f"    Hop {i}: {s}")

## 13. Hop Executor — Retrieve & Answer Each Sub-Question

Each hop retrieves evidence, considers findings from previous hops, and produces an intermediate answer.

In [ ]:
HOP_SYSTEM = """You answer a sub-question using the provided passages and any
prior findings. Be specific and factual. Cite passages as [PXX].
If the passages don't contain enough information, say what's missing."""

HOP_PROMPT = """SUB-QUESTION: {sub_question}

{prior_context}

PASSAGES:
{passages}

Answer this sub-question concisely using the passages. Cite [PXX] sources."""


def execute_hop(
    sub_question: str,
    prior_findings: list[dict],
    seen_passage_ids: set,
) -> dict:
    """Execute one retrieval hop: retrieve, answer, return findings."""
    # Enrich the query with prior findings for better retrieval
    enriched_query = sub_question
    if prior_findings:
        context_hint = "; ".join(f["answer"][:80] for f in prior_findings)
        enriched_query = f"{sub_question} (context: {context_hint})"

    # Retrieve (can optionally exclude already-seen passages)
    passages = retrieve(enriched_query, top_k=TOP_K, exclude_ids=list(seen_passage_ids))

    # Build prior context string
    prior_str = ""
    if prior_findings:
        prior_str = "PRIOR FINDINGS:\n" + "\n".join(
            f"  - {f['sub_question']}: {f['answer'][:120]}" for f in prior_findings
        )

    # Answer the sub-question
    context = "\n\n".join(f"[{p['passage_id']}] {p['text']}" for p in passages)
    answer = ask(
        HOP_PROMPT.format(
            sub_question=sub_question,
            prior_context=prior_str,
            passages=context,
        ),
        system=HOP_SYSTEM,
        temperature=TEMP_REASON,
    )

    new_ids = {p["passage_id"] for p in passages}
    return {
        "sub_question": sub_question,
        "answer": answer,
        "passages_retrieved": [p["passage_id"] for p in passages],
        "passage_ids": new_ids,
    }


print("Hop executor ready")
print("  Each hop: enriched query → retrieve → answer → intermediate finding")

## 14. Answer Synthesizer — Combine All Findings

After all hops, the synthesizer assembles a final answer from the chain of intermediate findings, with a clear reasoning trace.

In [ ]:
SYNTH_SYSTEM = """You synthesize a final answer from a chain of findings.
Show your reasoning: connect the dots between the intermediate findings.
Cite the original passages [PXX] used in each step."""

SYNTH_PROMPT = """ORIGINAL QUESTION: {question}

REASONING CHAIN:
{chain}

Synthesize a comprehensive final answer that:
1. Connects the findings into a coherent answer
2. Shows the reasoning chain (step by step)
3. Cites [PXX] sources
4. Acknowledges any gaps in the evidence"""


def synthesize_answer(question: str, findings: list[dict]) -> str:
    """Produce the final answer from accumulated hop findings."""
    chain = "\n\n".join(
        f"Step {i}: {f['sub_question']}\n"
        f"  Evidence from: {f['passages_retrieved']}\n"
        f"  Finding: {f['answer']}"
        for i, f in enumerate(findings, 1)
    )
    return ask(
        SYNTH_PROMPT.format(question=question, chain=chain),
        system=SYNTH_SYSTEM,
        temperature=TEMP_ANSWER,
    )


print("Synthesizer ready")

## 15. Full Multi-Hop Pipeline

In [ ]:
def multi_hop_rag(question: str, verbose: bool = True) -> dict:
    """Full multi-hop pipeline: decompose → hop 1..N → synthesize."""
    t0 = time.time()

    # Step 1: Decompose
    sub_questions = decompose_question(question)
    if verbose:
        print(f"  Decomposed into {len(sub_questions)} sub-questions")

    # Step 2: Execute hops
    findings = []
    seen_ids = set()

    for i, sq in enumerate(sub_questions, 1):
        hop_result = execute_hop(sq, findings, seen_ids)
        findings.append(hop_result)
        seen_ids.update(hop_result["passage_ids"])

        if verbose:
            print(f"  Hop {i}: {sq[:55]}...")
            print(f"         → [{', '.join(hop_result['passages_retrieved'])}]")
            print(f"         → {hop_result['answer'][:80]}...")

    # Step 3: Synthesize
    final_answer = synthesize_answer(question, findings)

    all_passages = []
    for f in findings:
        all_passages.extend(f["passages_retrieved"])

    return {
        "method": "multi_hop",
        "question": question,
        "sub_questions": sub_questions,
        "findings": findings,
        "answer": final_answer,
        "all_passages": list(dict.fromkeys(all_passages)),  # unique, ordered
        "hops": len(findings),
        "time_s": time.time() - t0,
    }


print("Multi-hop pipeline ready")
print("  decompose → hop(1) → hop(2) → ... → synthesize")

## 16. Multi-Hop Demo

In [ ]:
demo_q = "What caching strategy works best with the deployment pattern Netflix uses?"

print(f"MULTI-HOP Q&A")
print(f"Q: {demo_q}")
print("=" * 90)

multi = multi_hop_rag(demo_q, verbose=True)

print(f"\n{'='*90}")
print("FINAL ANSWER:")
wrap_print(multi["answer"])

print(f"\n  Total passages used: {multi['all_passages']}")
print(f"  Hops: {multi['hops']}")
print(f"  Time: {multi['time_s']:.1f}s")

## 17. Side-by-Side: Single-Hop vs Multi-Hop

In [ ]:
compare_q = demo_q

# Single-hop (already computed)
# Multi-hop (already computed)

print(f"Q: {compare_q}")
print("\n" + "=" * 90)
print("SINGLE-HOP")
print(f"  Passages: {single['passages_used']}")
print(f"  Time: {single['time_s']:.1f}s")
print(f"  Answer:")
wrap_print("    " + single["answer"][:300])

print("\n" + "=" * 90)
print("MULTI-HOP")
print(f"  Passages: {multi['all_passages']}")
print(f"  Hops: {multi['hops']}")
print(f"  Time: {multi['time_s']:.1f}s")
print(f"  Reasoning chain:")
for i, f in enumerate(multi["findings"], 1):
    print(f"    Step {i}: {f['sub_question'][:60]}")
    print(f"            → {f['answer'][:70]}...")
print(f"  Final answer:")
wrap_print("    " + multi["answer"][:300])

---

# Part D — Advanced: Adaptive Multi-Hop

## 18. Adaptive Hop Controller

Instead of pre-decomposing the question, the **adaptive** approach decides dynamically at each step whether more evidence is needed or the question can be answered.

In [ ]:
CONTROLLER_SYSTEM = """You control a multi-hop Q&A system. Given the original question
and evidence gathered so far, decide the next action."""

CONTROLLER_PROMPT = """ORIGINAL QUESTION: {question}

EVIDENCE SO FAR:
{evidence}

Can you fully answer the question with the evidence above?

Return ONLY JSON:
{{"can_answer": true/false,
  "next_query": "search query for missing evidence (if can_answer is false)",
  "reasoning": "what's missing or why you can answer"}}"""


def adaptive_multi_hop(question: str, verbose: bool = True) -> dict:
    """Dynamically decide when to stop hopping."""
    t0 = time.time()
    findings = []
    seen_ids = set()
    hop = 0

    # Initial retrieval
    passages = retrieve(question, top_k=TOP_K)
    hop_answer = ask(
        f"QUESTION: {question}\n\nPASSAGES:\n" +
        "\n\n".join(f"[{p['passage_id']}] {p['text']}" for p in passages) +
        "\n\nBriefly answer what you can from these passages:",
        system=HOP_SYSTEM,
        temperature=TEMP_REASON,
    )
    findings.append({
        "hop": 0,
        "query": question,
        "answer": hop_answer,
        "passages_retrieved": [p["passage_id"] for p in passages],
    })
    seen_ids.update(p["passage_id"] for p in passages)

    if verbose:
        print(f"  Hop 0 (initial): {[p['passage_id'] for p in passages]}")
        print(f"         → {hop_answer[:80]}...")

    while hop < MAX_HOPS:
        hop += 1

        # Ask the controller
        evidence_str = "\n".join(
            f"  Hop {f['hop']}: {f['answer'][:120]}" for f in findings
        )
        ctrl_resp = ask(
            CONTROLLER_PROMPT.format(question=question, evidence=evidence_str),
            system=CONTROLLER_SYSTEM,
            temperature=0.0,
        )
        ctrl = parse_json(ctrl_resp)

        if ctrl and ctrl.get("can_answer", False):
            if verbose:
                print(f"  Controller: CAN ANSWER — {ctrl.get('reasoning', '?')[:60]}")
            break

        next_query = ctrl.get("next_query", question) if ctrl else question
        if verbose:
            print(f"  Controller: NEED MORE — query: {next_query[:55]}")

        # Retrieve more
        new_passages = retrieve(next_query, top_k=TOP_K, exclude_ids=list(seen_ids))
        if not new_passages:
            if verbose:
                print(f"  Hop {hop}: No new passages found, stopping")
            break

        context = "\n\n".join(f"[{p['passage_id']}] {p['text']}" for p in new_passages)
        prior_str = "PRIOR FINDINGS:\n" + evidence_str
        hop_answer = ask(
            f"SUB-QUESTION: {next_query}\n\n{prior_str}\n\nPASSAGES:\n{context}\n\n"
            "What additional facts does this evidence provide?",
            system=HOP_SYSTEM,
            temperature=TEMP_REASON,
        )

        findings.append({
            "hop": hop,
            "query": next_query,
            "answer": hop_answer,
            "passages_retrieved": [p["passage_id"] for p in new_passages],
        })
        seen_ids.update(p["passage_id"] for p in new_passages)

        if verbose:
            print(f"  Hop {hop}: {[p['passage_id'] for p in new_passages]}")
            print(f"         → {hop_answer[:80]}...")

    # Synthesize
    final_answer = synthesize_answer(question, [
        {"sub_question": f["query"], "answer": f["answer"],
         "passages_retrieved": f["passages_retrieved"], "passage_ids": set()}
        for f in findings
    ])

    all_pids = []
    for f in findings:
        all_pids.extend(f["passages_retrieved"])

    return {
        "method": "adaptive_multi_hop",
        "question": question,
        "answer": final_answer,
        "findings": findings,
        "all_passages": list(dict.fromkeys(all_pids)),
        "hops": len(findings),
        "time_s": time.time() - t0,
    }


print("Adaptive multi-hop pipeline ready")
print("  retrieve → controller decides → retrieve more or stop → synthesize")

In [ ]:
adapt_q = "How should you monitor a Kubernetes cluster running event-sourced microservices?"

print(f"ADAPTIVE MULTI-HOP")
print(f"Q: {adapt_q}")
print("=" * 90)

adaptive = adaptive_multi_hop(adapt_q, verbose=True)

print(f"\n{'='*90}")
print("FINAL ANSWER:")
wrap_print(adaptive["answer"])

print(f"\n  Passages: {adaptive['all_passages']}")
print(f"  Hops: {adaptive['hops']}, Time: {adaptive['time_s']:.1f}s")

---

# Part E — Evaluation

## 19. Multi-Hop Test Set

In [ ]:
EVAL_QUESTIONS = [
    # Bridge questions (A → B → C)
    {"question": "What caching strategy works best with the deployment pattern Netflix uses?",
     "type": "bridge",
     "expected_topics": ["deployment", "caching"]},

    {"question": "How should you handle message delivery guarantees in microservices that use event sourcing?",
     "type": "bridge",
     "expected_topics": ["microservices", "event_sourcing", "messaging"]},

    {"question": "What load balancing approach works best for APIs that use JWT authentication?",
     "type": "bridge",
     "expected_topics": ["jwt", "load_balancing", "rest_api"]},

    # Comparison questions (A + B, then compare)
    {"question": "Compare how Kafka and RabbitMQ handle message failures and retries.",
     "type": "comparison",
     "expected_topics": ["kafka", "rabbitmq", "messaging"]},

    {"question": "How does PostgreSQL's durability approach differ from Redis's persistence strategy?",
     "type": "comparison",
     "expected_topics": ["postgresql", "redis"]},

    # Compositional questions (A → B, match)
    {"question": "What testing strategy is recommended for services deployed with blue-green deployments in Kubernetes?",
     "type": "compositional",
     "expected_topics": ["testing", "deployment", "kubernetes"]},

    {"question": "What monitoring should you add when sharding a PostgreSQL database across multiple servers?",
     "type": "compositional",
     "expected_topics": ["databases", "postgresql", "monitoring"]},

    # Inferential (A + B + reasoning)
    {"question": "Why would you choose Kafka over RabbitMQ for an event-sourced architecture with CQRS?",
     "type": "inferential",
     "expected_topics": ["kafka", "rabbitmq", "event_sourcing"]},

    {"question": "What security considerations apply when exposing GraphQL APIs through a Kubernetes Ingress?",
     "type": "inferential",
     "expected_topics": ["graphql", "kubernetes", "security"]},
]

print(f"Evaluation: {len(EVAL_QUESTIONS)} multi-hop questions")
print(f"Types: {dict(Counter(q['type'] for q in EVAL_QUESTIONS))}")

## 20. Run Evaluation — Single-Hop vs Multi-Hop

In [ ]:
print("Running evaluation (LLM-intensive)...\n")

eval_results = []

for i, item in enumerate(EVAL_QUESTIONS, 1):
    q = item["question"]

    single_r = single_hop_rag(q)
    multi_r = multi_hop_rag(q, verbose=False)

    # Check topic coverage
    def topics_covered(passage_ids):
        passage_map = {p["id"]: p["topic"] for p in PASSAGES}
        return set(passage_map.get(pid, "") for pid in passage_ids)

    single_topics = topics_covered(single_r["passages_used"])
    multi_topics = topics_covered(multi_r["all_passages"])
    expected = set(item["expected_topics"])

    single_coverage = len(single_topics & expected) / len(expected)
    multi_coverage = len(multi_topics & expected) / len(expected)

    eval_results.append({
        "question": q,
        "type": item["type"],
        "expected_topics": item["expected_topics"],
        "single": single_r,
        "multi": multi_r,
        "single_coverage": single_coverage,
        "multi_coverage": multi_coverage,
        "single_passages": single_r["passages_used"],
        "multi_passages": multi_r["all_passages"],
    })

    print(f"  [{i}/{len(EVAL_QUESTIONS)}] [{item['type']:<14}] "
          f"topic coverage: single={single_coverage:.0%} multi={multi_coverage:.0%} "
          f"| hops={multi_r['hops']} | {q[:40]}...")

## 21. Results & Analysis

In [ ]:
print("EVALUATION RESULTS")
print("=" * 100)

# Topic coverage comparison
single_coverages = [r["single_coverage"] for r in eval_results]
multi_coverages = [r["multi_coverage"] for r in eval_results]

print(f"\n  TOPIC COVERAGE (% of expected topics found in retrieved passages)")
print(f"  {'Method':<15} {'Average':>10} {'Full Coverage':>15}")
print(f"  {'-'*42}")
print(f"  {'Single-hop':<15} {sum(single_coverages)/len(single_coverages):>9.0%} "
      f"{sum(1 for c in single_coverages if c == 1.0):>14}/{len(eval_results)}")
print(f"  {'Multi-hop':<15} {sum(multi_coverages)/len(multi_coverages):>9.0%} "
      f"{sum(1 for c in multi_coverages if c == 1.0):>14}/{len(eval_results)}")

# By question type
print(f"\n  COVERAGE BY QUESTION TYPE")
print(f"  {'Type':<14} {'Single':>8} {'Multi':>8} {'Improvement':>12}")
print(f"  {'-'*45}")
for qtype in ["bridge", "comparison", "compositional", "inferential"]:
    items = [r for r in eval_results if r["type"] == qtype]
    if items:
        sc = sum(r["single_coverage"] for r in items) / len(items)
        mc = sum(r["multi_coverage"] for r in items) / len(items)
        print(f"  {qtype:<14} {sc:>7.0%} {mc:>7.0%} {mc - sc:>+11.0%}")

In [ ]:
# Passage breadth analysis
print("PASSAGE RETRIEVAL BREADTH")
print("=" * 100)
print(f"\n  {'Question':<55} {'Single':>8} {'Multi':>8} {'Multi Hops':>11}")
print(f"  {'-'*85}")

for r in eval_results:
    q_short = textwrap.shorten(r["question"], 53, placeholder="...")
    sp_count = len(set(r["single_passages"]))
    mp_count = len(set(r["multi_passages"]))
    hops = r["multi"]["hops"]
    print(f"  {q_short:<53} {sp_count:>6} {mp_count:>8} {hops:>9}")

avg_single = sum(len(set(r["single_passages"])) for r in eval_results) / len(eval_results)
avg_multi = sum(len(set(r["multi_passages"])) for r in eval_results) / len(eval_results)
avg_hops = sum(r["multi"]["hops"] for r in eval_results) / len(eval_results)

print(f"\n  Averages: single={avg_single:.1f} passages, multi={avg_multi:.1f} passages, hops={avg_hops:.1f}")

## 22. LLM-as-Judge — Answer Quality

In [ ]:
JUDGE_SYSTEM = """You compare two answers to a complex question.
Judge which better connects multiple sources of evidence into a coherent answer.
Value reasoning chains and multi-source synthesis over surface-level answers."""

JUDGE_PROMPT = """QUESTION: {question}

ANSWER A (single-hop, one retrieval):
{answer_a}

ANSWER B (multi-hop, chained retrieval):
{answer_b}

Which answer better addresses the multi-part question? Return ONLY JSON:
{{"winner": "A" or "B" or "tie",
  "reasoning": "brief explanation",
  "quality_a": 1-5,
  "quality_b": 1-5}}"""


print("QUALITY COMPARISON: Single-Hop vs Multi-Hop")
print("=" * 90)

wins = {"single_hop": 0, "multi_hop": 0, "tie": 0}
quality_scores = {"single": [], "multi": []}

for r in eval_results:
    resp = ask(
        JUDGE_PROMPT.format(
            question=r["question"],
            answer_a=r["single"]["answer"][:400],
            answer_b=r["multi"]["answer"][:400],
        ),
        system=JUDGE_SYSTEM,
        temperature=0.0,
    )
    j = parse_json(resp) or {"winner": "tie"}

    winner = "single_hop" if j.get("winner") == "A" else "multi_hop" if j.get("winner") == "B" else "tie"
    wins[winner] += 1

    qa = j.get("quality_a", 3)
    qb = j.get("quality_b", 3)
    if isinstance(qa, (int, float)):
        quality_scores["single"].append(qa)
    if isinstance(qb, (int, float)):
        quality_scores["multi"].append(qb)

    q_short = textwrap.shorten(r["question"], 55, placeholder="...")
    print(f"  {q_short:<55} → {winner}")

print(f"\n  WINS: single={wins['single_hop']}, multi={wins['multi_hop']}, tie={wins['tie']}")

if quality_scores["single"]:
    avg_s = sum(quality_scores["single"]) / len(quality_scores["single"])
    avg_m = sum(quality_scores["multi"]) / len(quality_scores["multi"])
    print(f"  AVG QUALITY: single={avg_s:.1f}/5, multi={avg_m:.1f}/5")

## 23. Detailed Reasoning Chain Inspection

In [ ]:
# Show full reasoning chains for 2 representative questions
print("REASONING CHAIN INSPECTION")
print("=" * 100)

for r in eval_results[:2]:
    print(f"\nQ: {r['question']}")
    print(f"Type: {r['type']}")
    print(f"Expected topics: {r['expected_topics']}")
    print("-" * 80)

    print(f"  SINGLE-HOP:")
    print(f"    Passages: {r['single_passages']}")
    wrap_print("    " + r["single"]["answer"][:250])

    print(f"\n  MULTI-HOP ({r['multi']['hops']} hops):")
    for i, f in enumerate(r["multi"]["findings"], 1):
        print(f"    Step {i}: {f['sub_question'][:60]}")
        print(f"      Evidence: {f['passages_retrieved']}")
        print(f"      Finding: {f['answer'][:100]}...")
    print(f"    FINAL ANSWER:")
    wrap_print("    " + r["multi"]["answer"][:250])

    print()

---

# Part F — Analysis & Learning Notes

## 24. Reasoning Chain Design Patterns

### Pattern 1: Sequential Bridge
```
Q: "What caching works for Netflix's deployment?"
  Hop 1: What deployment does Netflix use?  → Canary deployment
  Hop 2: What caching fits canary deployment? → Cache warming, gradual rollout
  Synthesize: Connect the two
```

### Pattern 2: Parallel + Merge
```
Q: "Compare Kafka vs RabbitMQ for failure handling"
  Hop 1: How does Kafka handle failures?     → Retention, replay, consumer groups
  Hop 2: How does RabbitMQ handle failures?  → DLQ, redelivery, acknowledgments
  Synthesize: Compare side-by-side
```

### Pattern 3: Drill-Down
```
Q: "What monitoring for sharded PostgreSQL?"
  Hop 1: What is database sharding?          → Partitioning across servers
  Hop 2: What's specific to PostgreSQL?      → WAL, MVCC, Citus extension
  Hop 3: What monitoring for these?          → Four golden signals, PromQL
  Synthesize: Specific recommendations
```

### When Multi-Hop Doesn't Help

| Situation | Why Multi-Hop Adds No Value |
|-----------|----------------------------|
| Simple factual question | One passage has the full answer |
| Passages already cover everything | No bridging needed |
| Question is vague | Decomposition just creates more vague sub-questions |
| Knowledge base is too narrow | Extra hops find the same limited information |

## 25. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Always running multi-hop | Adds latency for simple questions that don't need it | Use a classifier to decide single vs multi-hop |
| Too many hops | Each hop adds noise and LLM error accumulation | Cap at 3-4 hops; use adaptive controller |
| Ignoring previous hops' passages | Later hops re-retrieve the same passages | Track and exclude seen passage IDs |
| Not enriching hop queries | Sub-questions without context retrieve poorly | Include prior findings in the retrieval query |
| Pre-decomposing all questions | Decomposition quality varies; some questions don't decompose well | Adaptive approach: retrieve first, then decide if more is needed |
| No reasoning chain in final answer | User doesn't know how the answer was derived | Always show the step-by-step reasoning |

## 26. Production Improvement Ideas

1. **Hop classifier** — train a small model to decide single vs multi-hop before running the pipeline
2. **Parallel hops** — for comparison questions, run both branches simultaneously
3. **Hop caching** — cache intermediate findings for recurring sub-questions
4. **Confidence scoring** — rate each hop's finding confidence and stop early if high
5. **Cross-encoder reranking per hop** — rerank retrieved passages at each hop for precision
6. **Graph-based knowledge linking** — pre-build a passage relationship graph to guide hop queries
7. **Streaming synthesis** — start generating the final answer while later hops are still running

## 27. Exercises

### Exercise 1
Build a **hop confidence scorer**: after each hop, ask the LLM to rate its confidence (1-5) in the answer. If confidence is 5, skip remaining hops. Compare the results to always running all hops.

### Exercise 2
Implement **parallel decomposition**: for comparison questions, detect that two sub-questions are independent and can be answered in parallel. Measure the latency savings.

### Exercise 3
Add **contradiction detection**: when hop findings from multiple passages conflict, flag the contradiction and present both sides in the final answer instead of silently choosing one.

### Mini Challenge
Build a **routing layer** that classifies every incoming question as single-hop or multi-hop using the LLM, then routes to the appropriate pipeline. Measure whether routing accuracy exceeds 80% on a test set of mixed simple and complex questions.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Knowledge base: {len(PASSAGES)} passages, {len(set(p['topic'] for p in PASSAGES))} topics")
print(f"Evaluation:     {len(EVAL_QUESTIONS)} multi-hop questions, 4 types")
print()

# Coverage summary
avg_sc = sum(single_coverages) / len(single_coverages)
avg_mc = sum(multi_coverages) / len(multi_coverages)
print(f"Topic coverage:")
print(f"  Single-hop: {avg_sc:.0%}")
print(f"  Multi-hop:  {avg_mc:.0%} (+{avg_mc - avg_sc:.0%})")
print()
print(f"Judge results:")
print(f"  Single wins: {wins['single_hop']}/{len(eval_results)}")
print(f"  Multi wins:  {wins['multi_hop']}/{len(eval_results)}")
print(f"  Ties:        {wins['tie']}/{len(eval_results)}")
print()
print(f"Components built:")
print(f"  1. Question decomposer      — breaks complex Q into sub-Qs")
print(f"  2. Hop executor             — retrieve + answer per sub-Q")
print(f"  3. Answer synthesizer       — combine findings with reasoning chain")
print(f"  4. Adaptive controller      — dynamically decides when to stop hopping")
print(f"  5. Single-hop baseline      — standard single-retrieval RAG")
print(f"  6. LLM-as-judge evaluator   — quality comparison")

## 28. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Complex questions need multi-hop retrieval** — no single passage bridges topics like "caching for Netflix's deployment pattern" |
| 2 | **Question decomposition is the critical first step** — quality of sub-questions determines everything downstream |
| 3 | **Each hop enriches the next** — prior findings improve retrieval queries for subsequent hops |
| 4 | **Exclude seen passages** — prevents re-retrieving the same evidence and forces broader exploration |
| 5 | **Adaptive > pre-planned** — a controller that decides dynamically when to stop beats a fixed decomposition |
| 6 | **Topic coverage is a measurable proxy for quality** — multi-hop reliably covers more of the expected source topics |
| 7 | **Reasoning chains build trust** — showing step-by-step evidence accumulation makes answers interpretable |
| 8 | **Multi-hop adds latency** — 2-4x slower due to serial LLM calls; route simple questions to single-hop |
| 9 | **Error accumulates across hops** — each hop can introduce errors that propagate; keep hop count low |
| 10 | **Bridge, comparison, compositional, and inferential** are distinct multi-hop patterns requiring different decomposition strategies |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*